In [44]:
import pandas as pd
import numpy as np

In [45]:
from dataset_processor import DatasetProcessor

In [46]:
dp = DatasetProcessor.from_csv('zomato.csv')

In [47]:
dp.df.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


In [48]:
print(f"Dataset shape: {dp.df.shape}")

Dataset shape: (51717, 17)


In [49]:
dp.summary()

DATASET SUMMARY

Shape: 51717 rows x 17 columns

--- Column Info ---
                              Dtype  Non-Null   Null  Null%  Unique
url                          object     51717      0   0.00   51717
address                      object     51717      0   0.00   11495
name                         object     51717      0   0.00    8792
online_order                 object     51717      0   0.00       2
book_table                   object     51717      0   0.00       2
rate                         object     43942   7775  15.03      64
votes                         int64     51717      0   0.00    2328
phone                        object     50509   1208   2.34   14926
location                     object     51696     21   0.04      93
rest_type                    object     51490    227   0.44      93
dish_liked                   object     23639  28078  54.29    5271
cuisines                     object     51672     45   0.09    2723
approx_cost(for two people)  object     51371  

In [16]:
expected_columns = [
    'name', 'online_order', 'book_table',
    'rate', 'votes', 'phone', 'location', 'rest_type',
    'dish_liked', 'cuisines', 'approx_cost(for two people)'
]

In [17]:
schema_result = dp.validate_schema(expected_columns)

Schema validation passed. All expected columns are present.
Extra columns found: ['menu_item', 'listed_in(city)', 'url', 'reviews_list', 'listed_in(type)', 'address']


In [20]:
schema_result_fail = dp.validate_schema(['name', 'rate', 'votes', 'restaurant_establishment_date'])
schema_result_fail

Schema validation failed. Missing columns: ['restaurant_establishment_date']
Extra columns found: ['phone', 'approx_cost(for two people)', 'menu_item', 'cuisines', 'book_table', 'listed_in(city)', 'url', 'online_order', 'rest_type', 'location', 'dish_liked', 'reviews_list', 'listed_in(type)', 'address']


{'valid': False,
 'missing': ['restaurant_establishment_date'],
 'extra': ['phone',
  'approx_cost(for two people)',
  'menu_item',
  'cuisines',
  'book_table',
  'listed_in(city)',
  'url',
  'online_order',
  'rest_type',
  'location',
  'dish_liked',
  'reviews_list',
  'listed_in(type)',
  'address']}

In [40]:
expected_dtypes = {
    'votes': 'int64',
    'name': 'object',
    'online_order': 'object',
    'rate': 'object' 
}


In [23]:
dtype_result = dp.validate_dtypes(expected_dtypes)

Dtype validation passed.


In [51]:
dtype_res = dp.validate_dtypes({
    'votes': 'int64',
    'name': 'int64',
    'online_order': 'object',
    'rate': 'object' 
})

Dtype validation failed. Mismatches: {'name': {'expected': 'int64', 'actual': 'object'}}


In [24]:
print("Missing values per column:")
print(dp.df.isnull().sum())
print(f"\nTotal missing values: {dp.df.isnull().sum().sum()}")

Missing values per column:
url                                0
address                            0
name                               0
online_order                       0
book_table                         0
rate                            7775
votes                              0
phone                           1208
location                          21
rest_type                        227
dish_liked                     28078
cuisines                          45
approx_cost(for two people)      346
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64

Total missing values: 37700


In [25]:
dp_cleaned = dp.drop_missing(subset=['rate', 'cuisines'])
print(f"Before: {dp.df.shape[0]} rows")
print(f"After:  {dp_cleaned.df.shape[0]} rows")

Dropped 7786 rows with missing values.
Before: 51717 rows
After:  43931 rows


In [28]:
dp_filled = dp_cleaned.fill_missing(strategy='mode', columns=['dish_liked', 'rest_type'])

Filled missing values using strategy='mode' on 2 column(s).


In [29]:
print(f"dish_liked nulls: {dp_filled.df['dish_liked'].isnull().sum()}")
print(f"rest_type nulls:  {dp_filled.df['rest_type'].isnull().sum()}")

dish_liked nulls: 0
rest_type nulls:  0


In [30]:
dp_filled = dp_filled.fill_missing(strategy='mean', columns=['votes'])
print(f"votes nulls: {dp_filled.df['votes'].isnull().sum()}")

Filled missing values using strategy='mean' on 1 column(s).
votes nulls: 0


In [31]:
dp_deduped = dp_filled.drop_duplicates(subset=['name', 'address'])
print(f"Before: {dp_filled.df.shape[0]} rows")
print(f"After:  {dp_deduped.df.shape[0]} rows")

Dropped 33706 duplicate rows.
Before: 43931 rows
After:  10225 rows


In [32]:
dp_transformed = dp_deduped.add_column('rating', lambda df: df['rate'].astype(str).str.replace(r'\s*/\s*5', '', regex=True).apply(lambda x: float(x) if x.replace('.', '', 1).isdigit() else None))

Added column 'rating'.


In [33]:
dp_transformed.df[['name', 'rate', 'rating']].head(10)

,name,rate,rating
0,Jalsa,4.1/5,4.1
1,Spice Elephant,4.1/5,4.1
2,San Churro Cafe,3.8/5,3.8
3,Addhuri Udupi Bhojana,3.7/5,3.7
4,Grand Village,3.8/5,3.8
5,Timepass Dinner,3.8/5,3.8
6,Rosewood International Hotel - Bar & Restaurant,3.6/5,3.6
7,Onesta,4.6/5,4.6
8,Penthouse Cafe,4.0/5,4.0
9,Smacznego,4.2/5,4.2


In [37]:
dp_transformed = dp_transformed.add_column('cost', lambda df: pd.to_numeric(df['approx_cost(for two people)'].astype(str).str.replace(',', '', regex=False), errors='coerce'))

Added column 'cost'.


In [39]:
dp_transformed.df[['name','rating','cost']].head(10)

,name,rating,cost
0,Jalsa,4.1,800.0
1,Spice Elephant,4.1,800.0
2,San Churro Cafe,3.8,800.0
3,Addhuri Udupi Bhojana,3.7,300.0
4,Grand Village,3.8,600.0
5,Timepass Dinner,3.8,600.0
6,Rosewood International Hotel - Bar & Restaurant,3.6,800.0
7,Onesta,4.6,600.0
8,Penthouse Cafe,4.0,700.0
9,Smacznego,4.2,550.0
